# Символьное дифференцирование

## Порядок сдачи домашнего

Под каждое домашнее вы создаете отдельную ветку куда вносите все изменения в рамках домашнего. Как только домашнее готово - создаете пулл реквест (обратите внимание что в пулл реквесте должны быть отражены все изменения в рамках домашнего). Ревьювера назначаете из таблицы - https://docs.google.com/spreadsheets/d/1vK6IgEqaqXniUJAQOOspiL_tx3EYTSXW1cUrMHAZFr8/edit?gid=0#gid=0
Перед сдачей проверьте код, напишите тесты. Не забудьте про PEP8, например, с помощью flake8. Задание нужно делать в jupyter notebook.

**Дедлайн - 18 ноября 10:00**

Символьное дифференцирование это инструмент для автоматического вывода формул производных, который открывает возможности для анализа сложных функций, оптимизации процессов и работы с уравнениями. Мы уже на многих занятиях сталкивались с этой темой - давайте попробуем реализовать собственное!

Выполнил студент Андрющенко Ксения Сергеевна

## Выражение

Создадим основной класс `Expr`, от которого будут наследоваться различные типы выражений, такие как константы, переменные, суммы, произведения и другие. Класс должен содержать методы:
* `__call__`, который будет вычислять значение выражения, используя переданный ему контекст (словарь, связывающий имена переменных с их значениями).
* `d`, принимающий имя переменной, по которой требуется вычислить производную, и возвращающий выражение, представляющее производную по этой переменной.

Эти методы нужно будет переопределить в каждом из подклассов для корректного выполнения операций.

In [1]:
import unittest

In [2]:
class Expr:
    def __call__(self, **context):
        pass
    
    def d(self, wrt):
        pass

    def __pos__(self):
        return self

    def __neg__(self):
        return self * Const(-1)

    def __add__ (self, expr):
        return Sum(self, expr)

    def __mul__ (self, expr):
        return Product(self, expr)

    def __truediv__(self, expr):
        return Fraction(self, expr)
        
    def __sub__ (self, expr):
        return self + Const(-1) * expr

Создайте классы для двух видов выражений: `Const`, представляющий константу, и` Var`, представляющий переменную. Чтобы упростить использование, вместо обращения к конструкторам этих классов, будем использовать их однобуквенные сокращённые обозначения.

**Пример использования:**
```python
V = Var
C = Const

C(5)()
5
C(5).d(V("x"))()
0
V("x")(x=5)
5
V("x").d(V("y"))(x=5)
0
V("x").d(V("x"))(x=5)
1
```

In [3]:
class Const(Expr):
    def __init__(self, value):
        self.value = value

    def __call__ (self, **context):
        return self.value

    def d(self, wrt):
        return Const(0)
    

class Var(Expr):
    def __init__(self, var):
        self.var = var

    def __call__ (self, **context):
        return context[self.var]

    def d(self, wrt):
        if self.var == wrt.var:
            return Const(1)
        else:
            return Const(0)


In [4]:
V = Var
C = Const

In [5]:
class Test_Const_Var(unittest.TestCase):
    def test_1(self):
        self.assertEqual(C(5)(), 5)
    def test_2(self):
        self.assertEqual(C(5).d(V("x"))(), 0)
    def test_3(self):
        self.assertEqual(V("x")(x=5), 5)
    def test_4(self):
        self.assertEqual(V("x").d(V("y"))(x=5), 0)
    def test_5(self):
        self.assertEqual(V("x").d(V("x"))(x=5), 1)

In [6]:
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(Test_Const_Var))

.....
----------------------------------------------------------------------
Ran 5 tests in 0.008s

OK


<unittest.runner.TextTestResult run=5 errors=0 failures=0>

## Бинарные операции

Создайте классы для бинарных операций: `Sum`, `Product` и `Fraction`. Поскольку бинарные операции определяются двумя операндами, их конструктор будет одинаковым для всех этих классов. Поэтому его можно вынести в отдельный базовый класс, чтобы избежать дублирования кода.

In [7]:
class BinOp(Expr):
    def __init__(self, expr1, expr2):
        self.expr1, self.expr2 = expr1, expr2

Реализуйте `Sum` для суммирования, `Product` для умножения и `Fraction` для деления.

**Пример использования:**

```python
Sum(V("x"), Fraction(V("x"), V("y")))(x=5, y=2.5)
7.0
Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))(x=1, y=2)
3.5
Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("x"))(x=1, y=2)
-3.5
Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("y"))(x=1, y=2)
-1.25
```

In [8]:
class Sum(BinOp):
    def __call__(self, **context):
        a = self.expr1(**context)
        b = self.expr2(**context)
        return a + b

    def d(self, var):
        a = self.expr1.d(var)
        b = self.expr2.d(var)
        return a + b 

In [9]:
class Product(BinOp):
    def __call__(self, **context):
        a = self.expr1(**context)
        b = self.expr2(**context)
        return a * b

    def d(self, var):
        u = self.expr1
        du = self.expr1.d(var)
        v = self.expr2
        dv = self.expr2.d(var)
        return u * dv + v * du

In [10]:
class Fraction(BinOp):
    def __call__(self, **context):
        a = self.expr1(**context)
        b = self.expr2(**context)
        return a / b

    def d(self, var):
        u = self.expr1
        du = self.expr1.d(var)
        v = self.expr2
        dv = self.expr2.d(var)
        return (v * du - u * dv) / (v * v)

In [11]:
class Test_Binary_operations(unittest.TestCase):
    def test_1(self):
        self.assertEqual(Sum(V("x"), Fraction(V("x"), V("y")))(x=5, y=2.5), 7.0)
    def test_2(self):
        self.assertEqual(Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))(x=1, y=2), 3.5)
    def test_3(self):
        self.assertEqual(Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("x"))(x=1, y=2), -3.5)
    def test_4(self):
        self.assertEqual(Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("y"))(x=1, y=2), -1.25)

In [12]:
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(Test_Binary_operations))

....
----------------------------------------------------------------------
Ran 4 tests in 0.009s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

## Перегрузка операторов

Добавьте перегрузку операторов в базовых класс `Expr`. Обратите что в классах мы можем тоже заменить на использование операторов.
```python  
-e         e.__neg__()
+e         e.__pos__()
e1 + e2    e1.__add__(e2)
e1 - e2    e1.__sub__(e2)
e1 * e2    e1.__mul__(e2)
e1 / e2    e1.__truediv__(e2)
```

**Пример использования:**

```python
(V("x") * V("x") / V("y"))(x=5, y=2.5)
10.0
```

In [13]:
class Test_Overload(unittest.TestCase):
    def test_1(self):
        self.assertEqual((V("x") * V("x") / V("y"))(x=5, y=2.5), 10.0)

    def test_2(self):
        self.assertEqual((-C(5) + +C(5))(), 0)

In [14]:
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(Test_Overload))

..
----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

## Метод Ньютона-Рафсона

Напишите функцию `newton_raphson`, которая принимает дифференцируемую функцию  $f$  от переменной  $x$ , начальное приближение  $x_0$ , и положительное число  $\epsilon$ , задающее точность вычислений. Функция должна возвращать значение  $x$ , при котором  $f(x)$  становится равным нулю. Метод Ньютона-Рафсона выполняет итеративный поиск корня функции  $f(x)$ , начиная с начального значения  $x_0$ , и использует правило  
$$x_{n+1} = x_n - \frac{f(x_n)}{f{\prime}(x_n)}$$  
для обновления  $x$  на каждом шаге. Итерации продолжаются до тех пор, пока условие остановки  $|x_{n+1} - x_n| \leq \epsilon$  не будет выполнено.

**Пример использования:**

```python
x = Var("x")
f = Const(-5) * x * x * x * x * x + Const(3) * x + Const(2)
zero = newton_raphson(f, 0.5, eps=1e-4)
zero, f(x=zero)
(1.000000000001132, -2.490496697760136e-11)
```

In [15]:
def newton_raphson(func, x_start, eps=1e-4):
    x = V('x')
    prev_x = x_start
    x_n = x_start + 2 * eps
    g = x - (func / func.d(x))
    
    while abs(x_n - prev_x) > eps:
        prev_x = x_n
        x_n = g(x=x_n)
    return x_n
    

In [16]:
class Test_newton_raphson(unittest.TestCase):
    def test_1(self):
        x = Var("x")
        f = Const(-5) * x * x * x * x * x + Const(3) * x + Const(2)
        zero = newton_raphson(f, 0.5, eps=1e-4)
        self.assertEqual(zero, 1.000000000001132)
        self.assertEqual(f(x=zero), -2.490496697760136e-11)
    
    def test_2(self):
        x = Var("x")
        f = (Const(2) * x * x * x * x * x - Const(1)) / (x * x * x - x)
        zero = newton_raphson(f, 0.5, eps=1e-4)
        self.assertEqual(round(zero, 5), 0.87055)
        self.assertEqual(round(f(x=zero), 5), 0)

In [17]:
unittest.TextTestRunner().run(unittest.TestLoader().loadTestsFromTestCase(Test_newton_raphson))

..
----------------------------------------------------------------------
Ran 2 tests in 0.039s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>